In [ ]:
# ============================================================
# RESULTADOS MUNICIPAIS DO TESTE TEMPORAL DE 2025
#
# Objetivo:
# - usar SOMENTE os vínculos de 2025;
# - associar cada vínculo ao escore já produzido pelo RF final;
# - NÃO treinar novamente o modelo;
# - validar rigorosamente a ordem dos registros;
# - gerar resultados agregados por município;
# - manter TODOS os municípios na planilha;
# - sinalizar municípios com poucos vínculos.
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

from openpyxl import load_workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment
)


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA_BASE = (
    "/content/drive/MyDrive/TCC_2/dados/"
    "RAIS_BASE_MODELO_FINAL_V3"
)

PASTA_RESULTADOS = (
    "/content/drive/MyDrive/TCC_2/resultados"
)

PASTA_SAIDA = os.path.join(
    PASTA_RESULTADOS,
    "RESULTADOS_MUNICIPAIS_2025"
)

os.makedirs(PASTA_SAIDA, exist_ok=True)

COL_MUN = "Municipio_estabelecimento_codigo"
TARGET = "Y_doenca"

ANO = 2025

# cutoff escolhido em 2024 e congelado
CUTOFF = 0.7253175378


# ============================================================
# 2. LOCALIZAR ARTEFATOS JÁ GERADOS
# ============================================================

def localizar_arquivo(nome):
    candidatos = sorted(
        glob.glob(
            os.path.join(
                PASTA_RESULTADOS,
                "**",
                nome
            ),
            recursive=True
        )
    )

    if len(candidatos) == 0:
        raise FileNotFoundError(
            f"\nNão encontrei {nome} em:\n"
            f"{PASTA_RESULTADOS}"
        )

    if len(candidatos) > 1:
        print(
            f"\nATENÇÃO: encontrei mais de um arquivo "
            f"chamado {nome}:"
        )

        for i, arq in enumerate(candidatos, 1):
            print(f"{i}: {arq}")

        raise RuntimeError(
            "\nHá mais de um candidato. "
            "Informe manualmente o arquivo correto "
            "antes de prosseguir."
        )

    return candidatos[0]

ARQ_PROB = (
    "/content/drive/MyDrive/TCC_2/resultados/"
    "MODELO_FINAL_RF_RISCO_MUNICIPAL_V2/"
    "predicoes/prob_final_2025.npy"
)

ARQ_Y = (
    "/content/drive/MyDrive/TCC_2/resultados/"
    "MODELO_FINAL_RF_RISCO_MUNICIPAL_V2/"
    "predicoes/y_2025.npy"
)

print("\nArquivos utilizados:")
print("Probabilidades:", ARQ_PROB)
print("Y observado:   ", ARQ_Y)


# ============================================================
# 3. CARREGAR PROBABILIDADES E Y SALVOS
# ============================================================

prob = np.load(ARQ_PROB)
y_salvo = np.load(ARQ_Y)

# caso a probabilidade esteja salva com duas colunas
if prob.ndim == 2:

    if prob.shape[1] != 2:
        raise ValueError(
            f"Formato inesperado das probabilidades: "
            f"{prob.shape}"
        )

    prob = prob[:, 1]

prob = np.asarray(prob).ravel()
y_salvo = np.asarray(y_salvo).ravel().astype(int)

print("\nDimensões:")
print("prob_final_2025:", prob.shape)
print("y_2025:", y_salvo.shape)

if len(prob) != len(y_salvo):
    raise RuntimeError(
        "prob_final_2025 e y_2025 possuem "
        "quantidades diferentes de registros."
    )


# ============================================================
# 4. LOCALIZAR OS PARQUETS DE 2025
# ============================================================

arquivos_2025 = sorted(
    glob.glob(
        os.path.join(
            PASTA_BASE,
            "**",
            "RAIS_MODELO_FINAL_V3_2025_*.parquet"
        ),
        recursive=True
    )
)

if not arquivos_2025:
    raise FileNotFoundError(
        "Nenhum arquivo Parquet de 2025 encontrado."
    )

print(
    f"\nArquivos de 2025 encontrados: "
    f"{len(arquivos_2025)}"
)

for arq in arquivos_2025:
    print(" -", os.path.basename(arq))


# ============================================================
# 5. FUNÇÃO PARA NORMALIZAR MUNICÍPIO
# ============================================================

def normalizar_codigo_municipio(serie):

    s = pd.to_numeric(
        serie,
        errors="coerce"
    )

    return (
        s
        .astype("Int64")
        .astype("string")
        .str.zfill(6)
    )


# ============================================================
# 6. PROCESSAR 2025 NA MESMA ORDEM
#
# IMPORTANTE:
# o código compara Y_doenca reconstruído com y_2025.npy
# linha por linha.
#
# Se houver qualquer diferença, ele PARA.
# ============================================================

partes_agregadas = []

offset = 0
divergencias = 0
municipios_ausentes = 0


for num_arquivo, arquivo in enumerate(
    arquivos_2025,
    start=1
):

    print(
        f"\n[{num_arquivo}/{len(arquivos_2025)}] "
        f"{os.path.basename(arquivo)}"
    )

    pf = pq.ParquetFile(arquivo)

    for batch in pf.iter_batches(
        columns=[
            COL_MUN,
            TARGET
        ],
        batch_size=500_000
    ):

        df = batch.to_pandas()

        n = len(df)

        if offset + n > len(y_salvo):
            raise RuntimeError(
                "O número de registros reconstruídos "
                "ultrapassou o tamanho de y_2025.npy."
            )

        # ------------------------------
        # Y do parquet
        # ------------------------------

        y_batch = (
            pd.to_numeric(
                df[TARGET],
                errors="coerce"
            )
            .fillna(0)
            .gt(0)
            .astype("int8")
            .to_numpy()
        )

        # ------------------------------
        # Y salvo na mesma faixa
        # ------------------------------

        y_ref = y_salvo[
            offset:offset+n
        ]

        diferentes = np.sum(
            y_batch != y_ref
        )

        divergencias += diferentes

        if diferentes > 0:

            raise RuntimeError(
                f"\nERRO DE ALINHAMENTO!\n"
                f"Foram encontradas {diferentes} "
                f"divergências entre o Y reconstruído "
                f"e y_2025.npy no trecho iniciado em "
                f"{offset:,}.\n\n"
                f"A tabela NÃO será produzida."
            )

        # ------------------------------
        # probabilidades correspondentes
        # ------------------------------

        p_batch = prob[
            offset:offset+n
        ]

        pred_batch = (
            p_batch >= CUTOFF
        ).astype("int8")

        # ------------------------------
        # município
        # ------------------------------

        municipio = normalizar_codigo_municipio(
            df[COL_MUN]
        )

        municipios_ausentes += (
            municipio.isna().sum()
        )

        tmp = pd.DataFrame(
            {
                "Municipio_codigo": municipio,
                "Y": y_batch,
                "Escore": p_batch,
                "Pred": pred_batch
            }
        )

        # somente para agregação municipal
        tmp = tmp.dropna(
            subset=["Municipio_codigo"]
        )

        # componentes da matriz de confusão
        tmp["TP"] = (
            (tmp["Y"] == 1)
            &
            (tmp["Pred"] == 1)
        ).astype("int8")

        tmp["FP"] = (
            (tmp["Y"] == 0)
            &
            (tmp["Pred"] == 1)
        ).astype("int8")

        tmp["FN"] = (
            (tmp["Y"] == 1)
            &
            (tmp["Pred"] == 0)
        ).astype("int8")

        tmp["TN"] = (
            (tmp["Y"] == 0)
            &
            (tmp["Pred"] == 0)
        ).astype("int8")

        # para média correta depois
        tmp["Soma_escore"] = tmp["Escore"]

        ag = (
            tmp
            .groupby(
                "Municipio_codigo",
                as_index=False
            )
            .agg(
                N=("Y", "size"),
                Afastamentos=("Y", "sum"),
                Soma_escore=("Soma_escore", "sum"),
                Acima_cutoff=("Pred", "sum"),
                TP=("TP", "sum"),
                FP=("FP", "sum"),
                FN=("FN", "sum"),
                TN=("TN", "sum")
            )
        )

        partes_agregadas.append(ag)

        offset += n

        print(
            f"  processados: "
            f"{offset:,} / {len(y_salvo):,}",
            end="\r"
        )


print("\n")


# ============================================================
# 7. VALIDAÇÃO FINAL DA ORDEM
# ============================================================

if offset != len(y_salvo):

    raise RuntimeError(
        f"Foram reconstruídos {offset:,} registros, "
        f"mas y_2025 possui {len(y_salvo):,}."
    )

if divergencias != 0:

    raise RuntimeError(
        "Foram encontradas divergências de ordem."
    )

print("=" * 70)
print("VALIDAÇÃO DE ALINHAMENTO")
print("=" * 70)

print(
    f"Registros: {offset:,}"
)

print(
    f"Divergências de Y: "
    f"{divergencias}"
)

print(
    f"Municípios ausentes: "
    f"{municipios_ausentes:,}"
)

print(
    "\nALINHAMENTO LINHA A LINHA CONFIRMADO."
)


# ============================================================
# 8. CONFERIR MÉTRICAS GLOBAIS DE 2025
# ============================================================

pred_global = (
    prob >= CUTOFF
).astype(int)

ap = average_precision_score(
    y_salvo,
    prob
)

roc = roc_auc_score(
    y_salvo,
    prob
)

precision = precision_score(
    y_salvo,
    pred_global,
    zero_division=0
)

recall = recall_score(
    y_salvo,
    pred_global,
    zero_division=0
)

f1 = f1_score(
    y_salvo,
    pred_global,
    zero_division=0
)

bal_acc = balanced_accuracy_score(
    y_salvo,
    pred_global
)

tn, fp, fn, tp = confusion_matrix(
    y_salvo,
    pred_global
).ravel()

specificity = (
    tn / (tn + fp)
)

prevalencia = (
    y_salvo.mean()
)


print("\n" + "=" * 70)
print("MÉTRICAS GLOBAIS RECONSTRUÍDAS - 2025")
print("=" * 70)

print(
    f"Prevalência:         "
    f"{prevalencia:.4%}"
)

print(
    f"Average Precision:   "
    f"{ap:.4f}"
)

print(
    f"ROC-AUC:             "
    f"{roc:.4f}"
)

print(
    f"Precisão:            "
    f"{precision:.4%}"
)

print(
    f"Sensibilidade:       "
    f"{recall:.4%}"
)

print(
    f"Especificidade:      "
    f"{specificity:.4%}"
)

print(
    f"F1:                  "
    f"{f1:.4f}"
)

print(
    f"Acurácia balanceada: "
    f"{bal_acc:.4f}"
)

print(
    f"Cutoff:              "
    f"{CUTOFF:.10f}"
)


# ============================================================
# 9. CHECAGEM COM OS RESULTADOS DO TCC
# ============================================================

checagens = {
    "AP": (
        ap,
        0.263,
        0.005
    ),
    "ROC-AUC": (
        roc,
        0.888,
        0.005
    ),
    "Precisão": (
        precision,
        0.2435,
        0.01
    ),
    "Sensibilidade": (
        recall,
        0.5492,
        0.01
    ),
    "Especificidade": (
        specificity,
        0.9274,
        0.01
    )
}

problemas = []

for nome, (
    valor,
    esperado,
    tolerancia
) in checagens.items():

    if abs(
        valor - esperado
    ) > tolerancia:

        problemas.append(
            (
                nome,
                valor,
                esperado
            )
        )


if problemas:

    print(
        "\nATENÇÃO: algumas métricas ficaram "
        "fora da tolerância esperada:"
    )

    for nome, valor, esperado in problemas:

        print(
            f"{nome}: "
            f"obtido={valor:.5f}; "
            f"esperado≈{esperado:.5f}"
        )

    raise RuntimeError(
        "\nInterrompido para evitar produzir "
        "uma tabela com artefatos incorretos."
    )

else:

    print(
        "\nMétricas compatíveis com os "
        "resultados já apresentados no TCC."
    )


# ============================================================
# 10. CONSOLIDAR RESULTADOS POR MUNICÍPIO
# ============================================================

resultado = pd.concat(
    partes_agregadas,
    ignore_index=True
)

resultado = (
    resultado
    .groupby(
        "Municipio_codigo",
        as_index=False
    )
    .agg(
        N=("N", "sum"),
        Afastamentos=("Afastamentos", "sum"),
        Soma_escore=("Soma_escore", "sum"),
        Acima_cutoff=("Acima_cutoff", "sum"),
        TP=("TP", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum"),
        TN=("TN", "sum")
    )
)


# ============================================================
# 11. CALCULAR INDICADORES MUNICIPAIS DE 2025
# ============================================================

resultado["Prevalencia_observada"] = (
    resultado["Afastamentos"]
    /
    resultado["N"]
)

resultado["Escore_medio"] = (
    resultado["Soma_escore"]
    /
    resultado["N"]
)

resultado["Pct_acima_cutoff"] = (
    resultado["Acima_cutoff"]
    /
    resultado["N"]
)

resultado["Precisao_municipal"] = np.where(
    (
        resultado["TP"]
        +
        resultado["FP"]
    ) > 0,

    resultado["TP"]
    /
    (
        resultado["TP"]
        +
        resultado["FP"]
    ),

    np.nan
)

resultado["Sensibilidade_municipal"] = np.where(
    (
        resultado["TP"]
        +
        resultado["FN"]
    ) > 0,

    resultado["TP"]
    /
    (
        resultado["TP"]
        +
        resultado["FN"]
    ),

    np.nan
)


# ============================================================
# 12. IDENTIFICAR UF PELO CÓDIGO MUNICIPAL
# ============================================================

UF_POR_PREFIXO = {
    "11": ("RO", "Rondônia"),
    "12": ("AC", "Acre"),
    "13": ("AM", "Amazonas"),
    "14": ("RR", "Roraima"),
    "15": ("PA", "Pará"),
    "16": ("AP", "Amapá"),
    "17": ("TO", "Tocantins"),
    "21": ("MA", "Maranhão"),
    "22": ("PI", "Piauí"),
    "23": ("CE", "Ceará"),
    "24": ("RN", "Rio Grande do Norte"),
    "25": ("PB", "Paraíba"),
    "26": ("PE", "Pernambuco"),
    "27": ("AL", "Alagoas"),
    "28": ("SE", "Sergipe"),
    "29": ("BA", "Bahia"),
    "31": ("MG", "Minas Gerais"),
    "32": ("ES", "Espírito Santo"),
    "33": ("RJ", "Rio de Janeiro"),
    "35": ("SP", "São Paulo"),
    "41": ("PR", "Paraná"),
    "42": ("SC", "Santa Catarina"),
    "43": ("RS", "Rio Grande do Sul"),
    "50": ("MS", "Mato Grosso do Sul"),
    "51": ("MT", "Mato Grosso"),
    "52": ("GO", "Goiás"),
    "53": ("DF", "Distrito Federal"),
}


def identificar_uf(codigo):

    prefixo = str(codigo)[:2]

    return UF_POR_PREFIXO.get(
        prefixo,
        ("", "")
    )


identificacao = resultado[
    "Municipio_codigo"
].apply(identificar_uf)

resultado["UF"] = [
    x[0]
    for x in identificacao
]

resultado["Estado"] = [
    x[1]
    for x in identificacao
]


# ============================================================
# 13. TENTAR LOCALIZAR O LAYOUT DE MUNICÍPIOS DA RAIS
# ============================================================

layout_candidates = sorted(
    glob.glob(
        "/content/drive/MyDrive/TCC_2/**/"
        "RAIS_vinculos_layout2020.xlsx",
        recursive=True
    )
)

mapa_municipios = {}


if layout_candidates:

    ARQ_LAYOUT = layout_candidates[0]

    print(
        "\nLayout municipal localizado:"
    )

    print(ARQ_LAYOUT)

    layout = pd.read_excel(
        ARQ_LAYOUT,
        sheet_name="municipio",
        header=None
    )

    for valor in layout.iloc[:, 0].dropna():

        txt = str(valor)

        if ":" not in txt:
            continue

        codigo, nome_com_uf = txt.split(
            ":",
            1
        )

        codigo = codigo.strip().zfill(6)

        if "-" in nome_com_uf:

            _, nome = nome_com_uf.split(
                "-",
                1
            )

        else:

            nome = nome_com_uf

        mapa_municipios[
            codigo
        ] = nome.strip()


else:

    print(
        "\nATENÇÃO: layout de municípios "
        "não encontrado."
    )

    print(
        "Os resultados serão gerados "
        "com o código municipal."
    )


resultado["Municipio"] = (
    resultado[
        "Municipio_codigo"
    ].map(
        mapa_municipios
    )
)


# ============================================================
# 14. SINALIZAÇÃO DA QUANTIDADE DE VÍNCULOS
#
# NÃO excluímos nenhuma cidade.
# ============================================================

resultado["Faixa_amostra"] = np.select(
    [
        resultado["N"] < 100,

        (
            (resultado["N"] >= 100)
            &
            (resultado["N"] < 500)
        ),

        resultado["N"] >= 500
    ],

    [
        "N < 100 - interpretar com muita cautela",

        (
            "100 <= N < 500 - "
            "interpretar com cautela"
        ),

        "N >= 500"
    ],

    default=""
)


# ============================================================
# 15. RANKINGS
#
# O principal é pelo ESCORE MÉDIO.
#
# IMPORTANTE:
# escore não deve ser interpretado como probabilidade
# individual calibrada.
# ============================================================

resultado = resultado.sort_values(
    [
        "Escore_medio",
        "N"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

resultado["Posicao_escore"] = (
    np.arange(
        1,
        len(resultado) + 1
    )
)


# posição pela prevalência observada
ord_obs = (
    resultado[
        [
            "Municipio_codigo",
            "Prevalencia_observada",
            "N"
        ]
    ]
    .sort_values(
        [
            "Prevalencia_observada",
            "N"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

ord_obs[
    "Posicao_observada"
] = np.arange(
    1,
    len(ord_obs) + 1
)

resultado = resultado.merge(
    ord_obs[
        [
            "Municipio_codigo",
            "Posicao_observada"
        ]
    ],
    on="Municipio_codigo",
    how="left"
)


# ============================================================
# 16. ORGANIZAR COLUNAS
# ============================================================

resultado = resultado[
    [
        "Posicao_escore",
        "Posicao_observada",
        "Municipio_codigo",
        "Municipio",
        "UF",
        "Estado",
        "N",
        "Afastamentos",
        "Prevalencia_observada",
        "Escore_medio",
        "Acima_cutoff",
        "Pct_acima_cutoff",
        "TP",
        "FP",
        "FN",
        "TN",
        "Precisao_municipal",
        "Sensibilidade_municipal",
        "Faixa_amostra"
    ]
]


# ============================================================
# 17. CRIAR EXTREMOS PARA O TCC
#
# São criadas duas alternativas:
# A) N >= 500
# B) N >= 100
#
# Assim decidimos depois qual é mais adequada.
# ============================================================

def gerar_extremos(
    df,
    n_minimo
):

    base = df[
        df["N"] >= n_minimo
    ].copy()

    maiores = (
        base
        .sort_values(
            [
                "Escore_medio",
                "N"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(30)
        .copy()
    )

    maiores[
        "Extremo"
    ] = "30 maiores escores"

    menores = (
        base
        .sort_values(
            [
                "Escore_medio",
                "N"
            ],
            ascending=[
                True,
                False
            ]
        )
        .head(30)
        .copy()
    )

    menores[
        "Extremo"
    ] = "30 menores escores"

    return pd.concat(
        [
            maiores,
            menores
        ],
        ignore_index=True
    )


extremos_n500 = gerar_extremos(
    resultado,
    500
)

extremos_n100 = gerar_extremos(
    resultado,
    100
)


# ============================================================
# 18. RESUMO DAS FAIXAS DE AMOSTRA
# ============================================================

print("\n" + "=" * 70)
print("MUNICÍPIOS EM 2025")
print("=" * 70)

print(
    f"Total de municípios: "
    f"{len(resultado):,}"
)

print(
    "N >= 500:",
    (
        resultado["N"] >= 500
    ).sum()
)

print(
    "100 <= N < 500:",
    (
        (
            resultado["N"] >= 100
        )
        &
        (
            resultado["N"] < 500
        )
    ).sum()
)

print(
    "N < 100:",
    (
        resultado["N"] < 100
    ).sum()
)


# ============================================================
# 19. EXIBIR TOP 10 PARA CONFERÊNCIA
# ============================================================

print(
    "\n10 maiores escores médios "
    "entre municípios com N >= 500:"
)

print(
    resultado[
        resultado["N"] >= 500
    ][
        [
            "Municipio",
            "UF",
            "N",
            "Afastamentos",
            "Prevalencia_observada",
            "Escore_medio",
            "Pct_acima_cutoff"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 20. EXPORTAR CSV
# ============================================================

ARQ_CSV = os.path.join(
    PASTA_SAIDA,
    "resultados_municipais_2025_completo.csv"
)

resultado.to_csv(
    ARQ_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. EXPORTAR EXCEL
# ============================================================

ARQ_EXCEL = os.path.join(
    PASTA_SAIDA,
    "resultados_municipais_2025.xlsx"
)


with pd.ExcelWriter(
    ARQ_EXCEL,
    engine="openpyxl"
) as writer:

    resultado.to_excel(
        writer,
        sheet_name="Resultados_2025_todos",
        index=False
    )

    extremos_n500.to_excel(
        writer,
        sheet_name="Extremos_N500",
        index=False
    )

    extremos_n100.to_excel(
        writer,
        sheet_name="Extremos_N100",
        index=False
    )

    metricas = pd.DataFrame(
        {
            "Métrica": [
                "Prevalência",
                "Average Precision",
                "ROC-AUC",
                "Precisão",
                "Sensibilidade",
                "Especificidade",
                "F1",
                "Acurácia balanceada",
                "Cutoff"
            ],

            "Resultado": [
                prevalencia,
                ap,
                roc,
                precision,
                recall,
                specificity,
                f1,
                bal_acc,
                CUTOFF
            ]
        }
    )

    metricas.to_excel(
        writer,
        sheet_name="Metricas_globais_2025",
        index=False
    )

    guia = pd.DataFrame(
        [
            [
                "N",
                (
                    "Número de vínculos docentes "
                    "existentes no município em 2025."
                )
            ],
            [
                "Afastamentos",
                (
                    "Número de vínculos de 2025 com "
                    "registro administrativo de "
                    "afastamento por doença."
                )
            ],
            [
                "Prevalencia_observada",
                (
                    "Afastamentos observados divididos "
                    "pelo total de vínculos do município "
                    "em 2025."
                )
            ],
            [
                "Escore_medio",
                (
                    "Média dos escores produzidos pelo "
                    "Random Forest para os vínculos do "
                    "município. NÃO representa "
                    "probabilidade calibrada."
                )
            ],
            [
                "Acima_cutoff",
                (
                    "Quantidade de vínculos com escore "
                    ">= 0,7253175378."
                )
            ],
            [
                "Pct_acima_cutoff",
                (
                    "Percentual dos vínculos do município "
                    "classificados como positivos pelo "
                    "cutoff congelado em 2024."
                )
            ],
            [
                "TP",
                "Verdadeiros positivos."
            ],
            [
                "FP",
                "Falsos positivos."
            ],
            [
                "FN",
                "Falsos negativos."
            ],
            [
                "TN",
                "Verdadeiros negativos."
            ],
            [
                "Faixa_amostra",
                (
                    "Sinalização da quantidade de vínculos. "
                    "Todos os municípios permanecem na "
                    "planilha."
                )
            ]
        ],

        columns=[
            "Campo",
            "Interpretação"
        ]
    )

    guia.to_excel(
        writer,
        sheet_name="LEIA-ME",
        index=False
    )


# ============================================================
# 22. FORMATAÇÃO BÁSICA DO EXCEL
# ============================================================

wb = load_workbook(
    ARQ_EXCEL
)

cor_cabecalho = "1F4E78"
cor_branca = "FFFFFF"

for nome_aba in [
    "Resultados_2025_todos",
    "Extremos_N500",
    "Extremos_N100",
    "Metricas_globais_2025",
    "LEIA-ME"
]:

    ws = wb[nome_aba]

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:

        cell.fill = PatternFill(
            "solid",
            fgColor=cor_cabecalho
        )

        cell.font = Font(
            bold=True,
            color=cor_branca
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # largura automática limitada
    for col_cells in ws.columns:

        letra = col_cells[
            0
        ].column_letter

        tamanho = max(
            len(
                str(c.value)
            )
            if c.value is not None
            else 0
            for c in col_cells
        )

        ws.column_dimensions[
            letra
        ].width = min(
            max(
                tamanho + 2,
                10
            ),
            42
        )


# percentuais
for nome_aba in [
    "Resultados_2025_todos",
    "Extremos_N500",
    "Extremos_N100"
]:

    ws = wb[nome_aba]

    headers = {
        cell.value: cell.column
        for cell in ws[1]
    }

    for coluna in [
        "Prevalencia_observada",
        "Pct_acima_cutoff",
        "Precisao_municipal",
        "Sensibilidade_municipal"
    ]:

        if coluna in headers:

            c = headers[coluna]

            for linha in range(
                2,
                ws.max_row + 1
            ):

                ws.cell(
                    linha,
                    c
                ).number_format = "0.00%"


    if "Escore_medio" in headers:

        c = headers[
            "Escore_medio"
        ]

        for linha in range(
            2,
            ws.max_row + 1
        ):

            ws.cell(
                linha,
                c
            ).number_format = "0.0000"


wb.save(
    ARQ_EXCEL
)


# ============================================================
# 23. FINAL
# ============================================================

print("\n" + "=" * 70)
print("ARQUIVOS GERADOS")
print("=" * 70)

print(
    "\nExcel:"
)

print(
    ARQ_EXCEL
)

print(
    "\nCSV:"
)

print(
    ARQ_CSV
)

print(
    "\nPronto."
)